# Grover's Search Algorithm Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the "Grover's Search Algorithm" kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import Qubits


## Problem 1. Marking oracle for the prefix function

In this problem, the value of the function we're evaluating does not depend on the state of the qubits after qubit $P$. This means that we can just ignore them and consider only the qubits that matter - `x[:P]`.

Once we do that, the problem becomes much simpler: flip the state of the target qubit if the input qubits are in the given state. That's the definition of a controlled gate with arbitrary control pattern. In Workbench, we can apply that by converting the bit pattern $pattern$ into a little-endian integer and using it as part of the condition. 

In [ ]:
def oracle_starts_with(x: Qubits, y: Qubits, pattern: list[bool]) -> None:
    P = len(pattern)
    pattern_int = sum([2 ** ind if pattern[ind] else 0 for ind in range(P)])
    y.x(cond=x[:P] == pattern_int)

## Problem 2. Converting a marking oracle into a phase oracle

As we saw in the "Oracles" kata, we can allocate an additional qubit in the $\ket{-}$ state and use it as the target for our marking oracle.
This will "kick back" the $-1$ relative phase for the basis states $\ket{x}$ of the input register for which $f(x) = 1$, and leave the relative phases of the basis states for which $f(x) = 0$ unchanged.

In [ ]:
def marking_oracle_as_phase_oracle(marking_oracle: callable, x: Qubits) -> None:
    minus = Qubits(1, "minus", x.qpu)
    minus.x()
    minus.had()
    marking_oracle(x, minus)
    minus.had()
    minus.x()
    minus.release()

## Problem 3. Conditional phase flip

This operation is equivalent to the operation of flipping the sign of only the $\ket{0...0}$ basis state, with an additional global phase $-1$.
This means that you can implement it in two steps:

1. Flip the sign of the $\ket{0...0}$ basis state.
   You can do this using a convenient operation `reflect` which applies a relative phase to one basis state. If you apply it to the register `~x`, it will flip the sign of the $\ket{0...0}$ basis state.
2. Apply a global phase $-1$ to the whole state.
   Workbench doesn't have a built-in gate that applies a global phase, so the easiest way to achieve this is to pick a qubit and apply a sequence of gates $ZXZX$ to it.

In [ ]:
def conditional_phase_flip(x: Qubits) -> None:
    (~x).reflect()
    x[0].z()
    x[0].x()
    x[0].z()
    x[0].x()

> Copyright (c) 2026 PsiQuantum